# 6단계: QLoRA 파인튜닝 (조건 4) — 학습 (Colab)

목표: Colab T4(16GB)에서 Qwen2.5-Coder-7B-Instruct에 QLoRA 어댑터를 학습한다.
출력은 어댑터 하나이고, 예측 생성은 별도 노트북에서 vLLM으로 한다.

**조건 2·3과 다른 점 — AWQ는 학습할 수 없다.** 조건 2·3은 `Qwen2.5-Coder-7B-Instruct-AWQ`를 서빙했지만
AWQ는 추론 전용 양자화라 QLoRA를 얹을 수 없다. 그래서 학습은 **비양자화 원본 모델을 bitsandbytes NF4로**
불러와서 하고, 서빙은 조건 2·3과 동일한 AWQ 베이스에 어댑터만 얹는다(vLLM). 베이스 가중치가 조건 2·3과
같아야 "파인튜닝 효과"만 분리되기 때문이다. 학습(NF4)과 서빙(AWQ)의 양자화가 다른 건 알려진 절충이며
RESULTS.md에 기록한다.

**학습 설정** (로컬에서 잰 토큰 분포 기준)

| 항목 | 값 | 근거 |
|---|---|---|
| max_seq_len | 2048 | 초과 82개(1.2%)뿐. 중앙값 370, p90 856 |
| 초과 예제 | 학습에서 제외 | 잘리는 부분이 항상 정답 SQL이라 truncation은 불완전한 SQL을 가르침 |
| 정밀도 | fp16 | T4(Turing)는 bf16 미지원 |
| LoRA | r=64, alpha=16 | 브리프의 출발점 |
| 유효 배치 | 16 (1 × grad accum 16) | 16GB 한계 |

**사전 준비**: 런타임 유형 GPU(T4). `data/results/local_ft/train_messages.jsonl`(14MB) 업로드 필요.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bnb", bitsandbytes.__version__)

## 학습 데이터 업로드

로컬에서 `uv run python scripts/build_finetune_dataset.py`로 만든 `train_messages.jsonl`(7,000개, 14MB)을 올린다.
프롬프트는 조건 2의 추론 프롬프트와 바이트 단위로 동일하다 — 어댑터가 서빙될 형식 그대로 학습된다.

In [ ]:
from pathlib import Path

DATA = Path("train_messages.jsonl")
if not DATA.exists():
    from google.colab import files

    files.upload()  # data/results/local_ft/train_messages.jsonl 선택
assert DATA.exists(), "train_messages.jsonl 업로드 필요"
print(DATA.stat().st_size // 1024, "KB")

In [ ]:
import json

from datasets import Dataset
from transformers import AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"  # 비양자화 원본 (AWQ는 학습 불가)
MAX_SEQ_LEN = 2048

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

rows = [json.loads(line) for line in DATA.open(encoding="utf-8") if line.strip()]


# Tokenise one example and mask the prompt out of the labels. Done by hand rather
# than through TRL's dataset-format detection: given prompt/completion columns it
# still produced masked: 0, i.e. loss over the whole sequence including the schema.
# Explicit labels are inspectable, and the checks below fail loudly instead of
# quietly training on everything.
def encode(messages):
    prompt_text = tokenizer.apply_chat_template(
        messages[:2], tokenize=False, add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    if not full_text.startswith(prompt_text):
        return None
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
    if full_ids[: len(prompt_ids)] != prompt_ids:
        return None  # tokeniser merged across the boundary; skip rather than mislabel
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids) :]
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}


encoded, over_length, boundary = [], 0, 0
for r in rows:
    item = encode(r["messages"])
    if item is None:
        boundary += 1
    elif len(item["input_ids"]) > MAX_SEQ_LEN:
        # Drop rather than truncate: what gets cut is always the tail, i.e. the
        # gold SQL, so a truncated example would teach incomplete queries.
        over_length += 1
    else:
        encoded.append(item)

print(f"{len(encoded)}/{len(rows)} kept | {over_length} over {MAX_SEQ_LEN} tokens | {boundary} boundary mismatch")
dataset = Dataset.from_list(encoded)

# Verify the masking here, before the model is even loaded.
sample = encoded[0]
supervised = [x for x in sample["labels"] if x != -100]
assert 0 < len(supervised) < len(sample["labels"]), "prompt masking is not in effect"
print(f"first example: {len(sample['labels'])} tokens, {len(supervised)} supervised")
print("supervised text:", tokenizer.decode(supervised))

## 4bit 로드 + LoRA 부착

NF4 이중 양자화로 7B를 T4에 올린다. gradient checkpointing을 켜야 활성값 메모리가 맞는다.

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # T4 is Turing: no bf16
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map={"": 0}, dtype=torch.float16
)

# transformers 5 renamed torch_dtype to dtype and ignores the old name silently.
# When that happened the model loaded in the config's bfloat16, and fp16 training
# then died inside the GradScaler ("not implemented for 'BFloat16'") because T4 is
# Turing and has no bf16. Catch it here rather than at the first optimiser step.
dtypes = {p.dtype for p in model.parameters()}
print("parameter dtypes:", dtypes)
assert torch.bfloat16 not in dtypes, "model is bf16; fp16 training cannot work on a T4"

model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 학습

프롬프트가 아니라 **정답 SQL에만 손실을 건다** — 프롬프트까지 학습하면 어댑터 용량을 스키마 텍스트
암기에 쓰게 된다. 마스킹은 배치 단위로 한 번 더 검증하며, 실패하면 `assert`로 멈춘다.

TRL은 쓰지 않는다. `prompt`/`completion` 컬럼을 줘도 마스킹이 걸리지 않아(`masked: 0`)
위에서 라벨을 직접 만들었고, 여기서는 표준 `Trainer`만 쓴다. 인자 이름은 버전마다 사라지므로
(transformers 5는 `warmup_ratio`를 없앴다) 실제로 받는 필드만 걸러 넘기고 빠진 건 출력한다.

한 에폭은 약 433 스텝(6,918 / 16)이다. 중간에 세션이 끊겨도 되도록 50 스텝마다 체크포인트를 남긴다.

In [ ]:
from dataclasses import fields

from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

WANTED = dict(
    output_dir="qwen_spider_qlora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    # PEFT + checkpointing needs the non-reentrant path, else the backward pass
    # sees no grad-requiring input and errors out.
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    # Both spellings on purpose: transformers 5 dropped warmup_ratio and kept
    # warmup_steps, so the filter below keeps whichever this version has and
    # warmup does not silently vanish. 13 steps is 3% of the ~433-step epoch.
    warmup_ratio=0.03,
    warmup_steps=13,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

# TrainingArguments field names move between versions (transformers 5 dropped
# several), and passing one that no longer exists is a hard TypeError. Keep what
# this version accepts and say plainly what was dropped, rather than discovering
# the next missing name one traceback at a time.
accepted = {f.name for f in fields(TrainingArguments)}
used = {k: v for k, v in WANTED.items() if k in accepted}
missing = sorted(set(WANTED) - accepted)
if missing:
    print("NOT SUPPORTED by this version, dropped:", missing)
    hints = sorted(f for f in accepted if any(s in f for s in ("warm", "sched", "lr", "learning", "optim")))
    print("related fields this version does have:", hints)

args = TrainingArguments(**used)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    # Pads input_ids with the pad token and labels with -100, so padding is never
    # supervised. The dataset is already tokenised and masked above.
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
    processing_class=tokenizer,
)

# Re-check on a real collated batch: masking must survive padding and batching.
labels = next(iter(trainer.get_train_dataloader()))["labels"][0]
supervised = int((labels != -100).sum())
assert 0 < supervised < labels.numel(), "prompt masking is not in effect -- do not train"
print(f"batch tokens {labels.numel()}, supervised {supervised}, masked {labels.numel() - supervised}")
print("supervised text:", tokenizer.decode(labels[labels != -100]))

trainer.train(resume_from_checkpoint=None)  # 재개할 때는 True

## 어댑터 저장 + 다운로드

어댑터만 저장하므로 수백 MB 수준이다. 받은 파일은 로컬 `data/results/local_ft/adapter/`에 넣는다.

In [ ]:
import shutil

ADAPTER_DIR = "qwen_spider_qlora_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR)

from google.colab import files

files.download(f"{ADAPTER_DIR}.zip")